In [1]:
!pip3 install --upgrade --quiet langchain langchain-community langchain-experimental langchain-google-genai chromadb pypdf streamlit python-dotenv pyvis pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
!pip3 install --upgrade --quiet langchain-openai

In [ ]:
# Import Langchain modules
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Open AI
# from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

# Other modules and packages
import os
import tempfile
import streamlit as st
import pandas as pd
from dotenv import load_dotenv

In [ ]:
# Load the .env file
load_dotenv()
# Get API key from environment variable
# (make sure the key is present in .env file in the project directory)

# API_KEY = os.getenv("OPENAI_API_KEY") # os.environ.get("OPENAI_API_KEY")
API_KEY = os.getenv("GOOGLE_API_KEY")

## Define our LLM

In [ ]:
# llm = ChatOpenAI(model="gpt-4o-mini", api_key=API_KEY)
# llm.invoke("Tell me a joke about cats")

In [ ]:
import google.generativeai as genai

genai.configure(api_key=API_KEY)

for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gem

In [ ]:
from langchain_core.documents import Document

# from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

# llm = ChatOpenAI(temperature=0, model_name="gpt-4o")
model_free = "gemini-3.1-flash-lite-preview"
model_t = "gemini-2.5-flash"
model_latest = "gemini-flash-latest"
model_latest_lite = "gemini-flash-lite-latest"

llm = ChatGoogleGenerativeAI(
    model=model_latest,
    temperature=0.7,
#    max_tokens=None,
#    timeout=None,
    max_retries=2,
    api_key=API_KEY
)


# from langchain_experimental.graph_transformers import LLMGraphTransformer

# graph_transformer = LLMGraphTransformer(llm=llm)

## Process PDF document

### Load PDF document

In [ ]:
# Create 'data' directory if it doesn't exist
if not os.path.exists('data'):
    os.makedirs('data')

pdf_file_path = 'data/2402.11163.pdf'

# Download the PDF file only if it doesn't already exist
if not os.path.exists(pdf_file_path):
    print(f"Downloading {pdf_file_path}...")
    !wget -O {pdf_file_path} https://arxiv.org/pdf/2402.11163
else:
    print(f"'{pdf_file_path}' already exists. Skipping download.")

'data/2402.11163.pdf' already exists. Skipping download.


In [ ]:
loader = PyPDFLoader(pdf_file_path)
pages = loader.load()
# pages

Debug Purposes

In [ ]:
for i, page in enumerate(pages):
    print(f"--- Page {i+1} ---")
    print(f"Source: {page.metadata.get('source', 'N/A')}")
    print(f"Page Label: {page.metadata.get('page_label', 'N/A')}")
    print("Content (first 500 characters):\n" + page.page_content[:500] + "...")
    print("\n")

### Split document

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500,
                                            chunk_overlap=200,
                                            length_function=len,
                                            separators=["\n\n", "\n", " "])
chunks = text_splitter.split_documents(pages)

Debug Purposes

In [ ]:
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---")
    print(f"Source: {chunk.metadata.get('source', 'N/A')}")
    print(f"Page Label: {chunk.metadata.get('page_label', 'N/A')}")
    print("Content (first 500 characters):\n" + chunk.page_content[:500] + "...")
    print("\n")

### Create embeddings

In [ ]:
print("Listing all models with the supported method 'embedContent':")
for m in genai.list_models():
    # m.name, m.supported_generation_methods
    if "embedContent" in m.supported_generation_methods:
        print(f"Model Name: {m.name}, Supported Methods: {m.supported_generation_methods}")

Listing all models with the supported method 'embedContent':
Model Name: models/gemini-embedding-001, Supported Methods: ['embedContent', 'countTextTokens', 'countTokens', 'asyncBatchEmbedContent']
Model Name: models/gemini-embedding-2-preview, Supported Methods: ['embedContent', 'countTextTokens', 'countTokens', 'asyncBatchEmbedContent']


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

#def get_openai_embedding_function():
#    embeddings = OpenAIEmbeddings(
#        model="text-embedding-ada-002", openai_api_key=API_KEY
#    )
#    return embeddings

def get_gemini_embedding_function():
    embeddings = GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001",
        google_api_key=API_KEY
    )
    return embeddings

embedding_function = get_gemini_embedding_function()
test_vector = embedding_function.embed_query("cat")

In [ ]:
# from langchain.evaluation import load_evaluator
from langchain_classic.evaluation import load_evaluator

evaluator = load_evaluator(evaluator="embedding_distance",
                            embeddings=embedding_function)

evaluator.evaluate_strings(prediction="Amsterdam", reference="coffeeshop")

{'score': 0.19101626770842373}

In [ ]:
evaluator.evaluate_strings(prediction="Paris", reference="coffeeshop")

{'score': 0.24569902615435013}

### Create vector database

This cell defines a Python function called `create_vectorstore`. This function takes `chunks` (text segments from your document), an `embedding_function` (used to convert text into numerical vectors), and a `vectorstore_path` (where the vector database will be stored) as input. It then:

1.  Generates unique IDs for each text chunk.
2.  Filters out any duplicate chunks to ensure only unique content is processed.
3.  Creates a new Chroma vector database from these unique chunks and their embeddings, and persists it to the specified `vectorstore_path`.

In essence, it's responsible for taking your processed document chunks and storing them efficiently in a searchable database using their vector representations.

In [ ]:
import uuid

def create_vectorstore(chunks, embedding_function, vectorstore_path):

    # Create a list of unique ids for each document based on the content
    ids = [str(uuid.uuid5(uuid.NAMESPACE_DNS, doc.page_content)) for doc in chunks]

    # Ensure that only unique docs with unique ids are kept
    unique_ids = set()
    unique_chunks = []

    unique_chunks = []
    for chunk, id in zip(chunks, ids):
        if id not in unique_ids:
            unique_ids.add(id)
            unique_chunks.append(chunk);

    # Create a new Chroma database from the documents
    vectorstore = Chroma.from_documents(documents=unique_chunks,
                                        ids=list(unique_ids),
                                        embedding=embedding_function,
                                        persist_directory = vectorstore_path)

    return vectorstore

In [ ]:
# Create vectorstore
vectorstore = create_vectorstore(chunks=chunks,
                                 embedding_function=embedding_function,
                                 vectorstore_path="vectorstore_test")

## 2. Query for relevant data

This cell initializes the `vectorstore` by loading it from the `vectorstore_test` directory, using the previously defined `embedding_function`. It then creates a `retriever` from this `vectorstore`. The `retriever` is configured to use a `similarity` search, which means it will find document chunks that are most semantically similar to a given query.

In [ ]:
# Load vectorstore
vectorstore = Chroma(persist_directory="vectorstore_test", embedding_function=embedding_function)

retriever = vectorstore.as_retriever(search_type="similarity")

In [ ]:
relevant_chunks = retriever.invoke("What is the title of the paper?")
relevant_chunks

[Document(metadata={'source': 'data/2402.11163.pdf', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'keywords': '', 'trapped': '/False', 'subject': '', 'title': '', 'author': '', 'page': 0, 'total_pages': 16, 'creationdate': '2024-02-20T01:12:26+00:00', 'creator': 'LaTeX with hyperref', 'producer': 'pdfTeX-1.40.25', 'moddate': '2024-02-20T01:12:26+00:00', 'page_label': '1'}, page_content='KG-Agent: An Efficient Autonomous Agent Framework for Complex\nReasoning over Knowledge Graph\nJinhao Jiang1,3, Kun Zhou2,3, Wayne Xin Zhao1,3∗, Yang Song4∗,\nChen Zhu5, Hengshu Zhu5, Ji-Rong Wen1,2,3\n1Gaoling School of Artificial Intelligence, Renmin University of China.\n2School of Information, Renmin University of China.\n3Beijing Key Laboratory of Big Data Management and Analysis Methods.\n4NLP Center, BOSS Zhipin. 5Career Science Lab, BOSS Zhipin.\njiangjinhao@ruc.edu.cn, batmanfly@gmail.com\nAbstract\nIn this paper, we aim to improve

In [ ]:
# Prompt template
PROMPT_TEMPLATE = """
You are an expert in information extraction. Extract only subject-predicate-object triplets, which related to question, from the provided text.
Use the following pieces of retrieved context to answer
the question. If you don't know the answer, DON'T MAKE UP ANYTHING. Do not use markup. Use one line for each triplet with format: SUBJECT --> PREDICATE --> OBJECT

---

{context}

---

Answer the question based on the above context: {question}
"""

## 3. Generate responses

In [ ]:
# Concatenate context text
context_text = "\n\n---\n\n".join([doc.page_content for doc in relevant_chunks])

# Create prompt
prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
prompt = prompt_template.format(context=context_text,
                                question="What is the contribution of this paper? Used methods, dataset, materials? What is the research gap?")
# print(prompt)

In [ ]:
llm.invoke(prompt)

AIMessage(content=[{'type': 'text', 'text': 'Paper --> proposes --> KG-Agent framework\nKG-Agent --> improves --> reasoning ability of large language models over knowledge graphs\nKG-Agent --> enables --> small LLM to actively make decisions\nKG-Agent --> integrates --> LLM\nKG-Agent --> integrates --> multifunctional toolbox\nKG-Agent --> integrates --> KG-based executor\nKG-Agent --> integrates --> knowledge memory\nKG-Agent --> develops --> iteration mechanism\nIteration mechanism --> autonomously selects --> tool\nIteration mechanism --> updates --> memory\nPaper --> leverages --> program language\nProgram language --> formulates --> multi-hop reasoning process\nPaper --> synthesizes --> code-based instruction dataset\nCode-based instruction dataset --> fine-tunes --> base LLM\nExperiment --> tunes --> LLaMA-7B\nExperiment --> uses --> 10K samples\nResearch gap --> is --> reasoning ability of large language models over knowledge graphs to answer complex questions\nExisting methods 

### Using Langchain Expression Language

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
rag_chain = (
            {"context": retriever | format_docs, "question": RunnablePassthrough()}
            | prompt_template
            | llm
        )
rag_chain.invoke("What is the contribution of this paper? Used methods, dataset, materials? What is the research gap?")

AIMessage(content=[{'type': 'text', 'text': "Approach --> makes --> three major technical contributions\nApproach --> enables --> autonomous reasoning approaches\nAutonomous reasoning approaches --> make --> decisions without human assistance\nApproach --> enables --> 7B LLM to effectively perform complex reasoning\nApproach --> eliminates --> reliance on close-sourced LLM APIs\nApproach --> extends --> LLM capacity to manipulate structured data\nApproach --> curates --> multifunctional toolbox\nMultifunctional toolbox --> enables --> LLM to perform discrete or advanced operations\nResearch gap --> is --> reliance on human assistance during reasoning\nResearch gap --> is --> reliance on close-sourced LLM APIs for complex reasoning\nResearch gap --> is --> inability of small models to perform complex reasoning\nApproach --> uses --> 7B LLM\nApproach --> uses --> structured data\nI don't know the specific dataset used.", 'extras': {'signature': 'ErlvCrZvAQw51sdk82WKCQsmZLqEyCjYSF003ce/zE

### Defining a Pydantic model for Triplet Extraction

#### Generate structured responses

Since we're not using the `LLMGraphTransformer`, we can define a custom Pydantic model to guide the LLM to extract structured triplets (subject, predicate, object) from the text. This allows us to get the desired structured output directly from the LLM.

In [ ]:
class Triplet(BaseModel):
    subject: str = Field(description="The subject of the triplet")
    predicate: str = Field(description="The predicate (relationship) of the triplet")
    obj: str = Field(description="The object of the triplet")

class TripletExtraction(BaseModel):
    triplets: list[Triplet] = Field(description="A list of extracted triplets (subject, predicate, obj)")

### Extracting Triplet Information

Now we'll use the defined `TripletExtraction` Pydantic model with `llm.with_structured_output` to extract triplets from a selected chunk of the document. This allows the LLM to parse the text and return the information in a structured JSON format according to our schema.

In [ ]:
# Let's use the first chunk to demonstrate triplet extraction
sample_chunk = chunks[1].page_content

# Define a prompt specifically for triplet extraction
triplet_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in information extraction. Extract all possible subject-predicate-object triplets from the provided text."),
    ("human", "Extract triplets from the following text:\n\n{text}")
])

# Create a new chain for triplet extraction with structured output
triplet_extraction_chain = (
    {"text": RunnablePassthrough()}
    | triplet_prompt_template
    | llm.with_structured_output(TripletExtraction, strict=True)
)

# Invoke the chain on the sample chunk
extracted_triplets = triplet_extraction_chain.invoke(sample_chunk)

# Display the extracted triplets
for triplet in extracted_triplets.triplets:
    print(f"Subject: {triplet.subject}, Predicate: {triplet.predicate}, Object: {triplet.obj}")


Subject: LLaMA-7B, Predicate: tuned with, Object: 10K samples
Subject: LLaMA-7B, Predicate: outperforms, Object: state-of-the-art methods
Subject: Our code and data, Predicate: will be released, Object: publicly
Subject: large language models, Predicate: have, Object: limited capacities in solving complex tasks
Subject: Knowledge graph, Predicate: stores, Object: massive knowledge triples
Subject: Knowledge graph, Predicate: complements, Object: LLMs with external knowledge
Subject: Recent work, Predicate: adopts, Object: retrieval-augmented methods
Subject: Recent work, Predicate: adopts, Object: synergy-augmented methods
Subject: retrieval-augmented approach, Predicate: retrieves, Object: task-related triples
Subject: retrieval-augmented approach, Predicate: serializes, Object: task-related triples
Subject: synergy-augmented approach, Predicate: designs, Object: information interaction mechanism
Subject: synergy-augmented methods, Predicate: benefit from, Object: structured search on

### Querying for Structured Triplets with RAG

Now, let's create a RAG chain that queries for relevant document chunks and then extracts structured triplets from the combined content of these chunks. This allows us to retrieve specific information in a structured format based on a natural language question.

In [ ]:
# Define a prompt for RAG-based triplet extraction
rag_triplet_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in information extraction. Based on the provided context, extract all possible subject-predicate-object triplets that directly answer or are highly relevant to the user's question."),
    ("human", "Context: {context}\n\nQuestion: {question}\n\nExtract triplets:")
])

# Create a RAG chain for structured triplet extraction
rag_triplet_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_triplet_prompt_template
    | llm.with_structured_output(TripletExtraction, strict=True)
)

# Example query for triplets
query_triplets = rag_triplet_chain.invoke("What is the contribution of this paper? Used methods, dataset, materials? What is the research gap?")

print("Extracted Triplets based on Query:")
for triplet in query_triplets.triplets:
    print(f"Subject: {triplet.subject}, Predicate: {triplet.predicate}, Object: {triplet.obj}")

Extracted Triplets based on Query:
Subject: This paper, Predicate: makes, Object: three major technical contributions
Subject: This paper, Predicate: extends, Object: LLM capacity to manipulate structured data
Subject: This paper, Predicate: curates, Object: a multifunctional toolbox
Subject: This paper, Predicate: enables, Object: LLM to perform discrete or advanced operations
Subject: The approach, Predicate: enables, Object: relatively small models to effectively perform complex reasoning
Subject: The approach, Predicate: eliminates, Object: reliance on close-sourced LLM APIs
Subject: The approach, Predicate: utilizes, Object: autonomous reasoning approaches
Subject: Research gap, Predicate: includes, Object: autonomous reasoning approaches that can actively make decisions without human assistance
Subject: Research gap, Predicate: includes, Object: enabling small models to perform complex reasoning without close-sourced LLM APIs
Subject: 7B LLM, Predicate: is an example of, Object: 